In [66]:
!pip install pyspark

In [67]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [68]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
    TimestampType
)

In [69]:
spark = SparkSession.builder \
    .appName("DVD_Rental_Analysis") \
    .getOrCreate()

In [95]:
film_schema = StructType([
    StructField("film_id", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("description", StringType(), True),
    StructField("release_year", IntegerType(), True),
    StructField("language_id", IntegerType(), True),
    StructField("rental_duration", IntegerType(), True),
    StructField("rental_rate", DoubleType(), True),
    StructField("length", IntegerType(), True),
    StructField("replacement_cost", DoubleType(), True),
    StructField("rating", StringType(), True)
])

category_schema = StructType([
    StructField("category_id", IntegerType(), True),
    StructField("name", StringType(), True)
])

film_category_schema = StructType([
    StructField("film_id", IntegerType(), True),
    StructField("category_id", IntegerType(), True)
])

actor_schema = StructType([
    StructField("actor_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True)
])

film_actor_schema = StructType([
    StructField("actor_id", IntegerType(), True),
    StructField("film_id", IntegerType(), True)
])

inventory_schema = StructType([
    StructField("inventory_id", IntegerType(), True),
    StructField("film_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True)
])

rental_schema = StructType([
    StructField("rental_id", IntegerType(), True),
    StructField("rental_date", TimestampType(), True),
    StructField("inventory_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("return_date", TimestampType(), True),
    StructField("staff_id", IntegerType(), True),
    StructField("last_update", StringType(), True)
])

payment_schema = StructType([
    StructField("payment_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("staff_id", IntegerType(), True),
    StructField("rental_id", IntegerType(), True),
    StructField("amount", DoubleType(), True)
])

customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("address_id", IntegerType(), True),
    StructField("activebool", StringType(), True),
    StructField("create_date", StringType(), True),
    StructField("last_update", StringType(), True),
    StructField("active", IntegerType(), True)
])

address_schema = StructType([
    StructField("address_id", IntegerType(), True),
    StructField("address", StringType(), True),
    StructField("address2", StringType(), True),
    StructField("district", StringType(), True),
    StructField("city_id", IntegerType(), True),
    StructField("postal_code", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("last_update", StringType(), True)
])

city_schema = StructType([
    StructField("city_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("country_id", IntegerType(), True),
    StructField("last_update", StringType(), True)
])

In [71]:
from google.colab import files
uploaded = files.upload()

Saving payment.csv to payment (2).csv
Saving film_category.csv to film_category (2).csv
Saving address.csv to address (2).csv
Saving film.csv to film (2).csv
Saving category.csv to category (2).csv
Saving inventory.csv to inventory (2).csv
Saving customer.csv to customer (2).csv
Saving city.csv to city (2).csv
Saving actor.csv to actor (2).csv
Saving rental.csv to rental (2).csv
Saving film_actor.csv to film_actor (2).csv


In [105]:
film = spark.read.csv("film.csv", header=True, schema=film_schema)
category = spark.read.csv("category.csv", header=True, schema=category_schema)
film_category = spark.read.csv("film_category.csv", header=True, schema=film_category_schema)
actor = spark.read.csv("actor.csv", header=True, schema=actor_schema)
film_actor = spark.read.csv("film_actor.csv", header=True, schema=film_actor_schema)
inventory = spark.read.csv("inventory.csv", header=True, schema=inventory_schema)
rental = spark.read.csv("rental.csv", header=True, schema=rental_schema)
payment = spark.read.csv("payment.csv", header=True, schema=payment_schema)
customer = spark.read.csv("customer.csv", header=True, schema=customer_schema)
address = spark.read.csv("address.csv", header=True, schema=address_schema)
city = spark.read.csv("city.csv", header=True, schema=city_schema)

In [73]:
#  NUMBER OF MOVIES IN EACH CATEGORY
movies_per_category = (
    film_category
    .join(category, on="category_id", how="inner")
    .groupBy("name")
    .agg(
        F.count("film_id").alias("movie_count")
    )
    .orderBy(F.desc("movie_count"))
)

print("1. Number of movies in each category")
movies_per_category.show(truncate=False)

1. Number of movies in each category
+-----------+-----------+
|name       |movie_count|
+-----------+-----------+
|Sports     |74         |
|Foreign    |73         |
|Family     |69         |
|Documentary|68         |
|Animation  |66         |
|Action     |64         |
|New        |63         |
|Drama      |62         |
|Games      |61         |
|Sci-Fi     |61         |
|Children   |60         |
|Comedy     |58         |
|Travel     |57         |
|Classics   |57         |
|Horror     |56         |
|Music      |51         |
+-----------+-----------+



In [74]:
# 2. TOP 10 ACTORS WHOSE MOVIES RENTED MOST
top_actors = (
    rental
    .join(inventory, on="inventory_id", how="inner")
    .join(film_actor, on="film_id", how="inner")
    .join(actor, on="actor_id", how="inner")
    .groupBy(
        "actor_id",
        "first_name",
        "last_name"
    )
    .agg(
        F.count("rental_id").alias("rental_count")
    )
    .orderBy(F.desc("rental_count"))
    .limit(10)
)

print("2. Top 10 actors whose movies rented the most")
top_actors.show(truncate=False)

2. Top 10 actors whose movies rented the most
+--------+----------+-----------+------------+
|actor_id|first_name|last_name  |rental_count|
+--------+----------+-----------+------------+
|107     |GINA      |DEGENERES  |753         |
|181     |MATTHEW   |CARREY     |678         |
|198     |MARY      |KEITEL     |674         |
|144     |ANGELA    |WITHERSPOON|654         |
|102     |WALTER    |TORN       |640         |
|60      |HENRY     |BERRY      |612         |
|150     |JAYNE     |NOLTE      |611         |
|37      |VAL       |BOLGER     |605         |
|23      |SANDRA    |KILMER     |604         |
|90      |SEAN      |GUINESS    |599         |
+--------+----------+-----------+------------+



In [75]:
# 3. CATEGORY WITH HIGHEST REVENUE
category_revenue = (
    payment
    .join(rental, on="rental_id", how="inner")
    .join(inventory, on="inventory_id", how="inner")
    .join(film_category, on="film_id", how="inner")
    .join(category, on="category_id", how="inner")
    .groupBy("name")
    .agg(
        F.round(
            F.sum("amount"), 2
        ).alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)

print("3. Category with highest revenue")
category_revenue.show(1, truncate=False)

3. Category with highest revenue
+------+-------------+
|name  |total_revenue|
+------+-------------+
|Sports|5314.21      |
+------+-------------+
only showing top 1 row


In [76]:
#4. MOVIES NOT IN INVENTORY

movies_not_in_inventory = (
    film
    .join(inventory, on="film_id", how="left_anti" )
    .select("title")
)

print("4. Movies not in inventory")
movies_not_in_inventory.show(truncate=False)

4. Movies not in inventory
+----------------------+
|title                 |
+----------------------+
|ALICE FANTASIA        |
|APOLLO TEEN           |
|ARGONAUTS TOWN        |
|ARK RIDGEMONT         |
|ARSENIC INDEPENDENCE  |
|BOONDOCK BALLROOM     |
|BUTCH PANTHER         |
|CATCH AMISTAD         |
|CHINATOWN GLADIATOR   |
|CHOCOLATE DUCK        |
|COMMANDMENTS EXPRESS  |
|CROSSING DIVORCE      |
|CROWDS TELEMARK       |
|CRYSTAL BREAKING      |
|DAZED PUNK            |
|DELIVERANCE MULHOLLAND|
|FIREHOUSE VIETNAM     |
|FLOATS GARDEN         |
|FRANKENSTEIN STRANGER |
|GLADIATOR WESTWARD    |
+----------------------+
only showing top 20 rows


In [96]:
# 5. TOP 3 ACTORS IN CHILDREN CATEGORY

children_movies = (
    film_actor
    .join(film_category, on="film_id", how="inner")
    .join(category, on="category_id", how="inner")
    .filter(F.col("name") == "Children")
    .join(actor, on="actor_id", how="inner")
    .groupBy("actor_id", "first_name","last_name")
    .agg(F.countDistinct("film_id").alias("movie_count"))
)

window_spec = Window.orderBy(F.desc("movie_count"))

ranked_children_actors = (children_movies
        .withColumn(
        "rank",
        F.dense_rank().over(window_spec)
    )
    .filter(F.col("rank") <= 3)
    .orderBy("rank")
)

print("5. Top actors in Children category")
ranked_children_actors.show(truncate=False)

5. Top actors in Children category
+--------+----------+---------+-----------+----+
|actor_id|first_name|last_name|movie_count|rank|
+--------+----------+---------+-----------+----+
|17      |HELEN     |VOIGHT   |7          |1   |
|127     |KEVIN     |GARLAND  |5          |2   |
|80      |RALPH     |CRUZ     |5          |2   |
|66      |MARY      |TANDY    |5          |2   |
|140     |WHOOPI    |HURT     |5          |2   |
|81      |SCARLETT  |DAMON    |4          |3   |
|109     |SYLVESTER |DERN     |4          |3   |
|23      |SANDRA    |KILMER   |4          |3   |
|187     |RENEE     |BALL     |4          |3   |
|92      |KIRSTEN   |AKROYD   |4          |3   |
|173     |ALAN      |DREYFUSS |4          |3   |
|101     |SUSAN     |DAVIS    |4          |3   |
|150     |JAYNE     |NOLTE    |4          |3   |
|13      |UMA       |WOOD     |4          |3   |
|131     |JANE      |JACKMAN  |4          |3   |
|58      |CHRISTIAN |AKROYD   |4          |3   |
|142     |JADA      |RYDER    |4  

In [112]:
 # 6. CITIES WITH ACTIVE AND INACTIVE CUSTOMERS

city_customers = (
    customer
    .join(address, on="address_id", how="inner")
    .join(city, on="city_id", how="inner")
    .groupBy("city")
    .agg(
        F.sum(
            F.when(
                F.col("activebool") == True,
                1
            ).otherwise(0)
        ).alias("active_customers"),

        F.sum(
            F.when(
                F.col("activebool") == False,
                1
            ).otherwise(0)
        ).alias("inactive_customers")
    )
    .orderBy(
        F.desc("inactive_customers")
    )
)

print("6. Cities with active and inactive customers")

city_customers.show(
    truncate=False
)

6. Cities with active and inactive customers
+------------------+----------------+------------------+
|city              |active_customers|inactive_customers|
+------------------+----------------+------------------+
|A Corua (La Corua)|1               |0                 |
|Fengshan          |1               |0                 |
|Myingyan          |1               |0                 |
|Chisinau          |1               |0                 |
|Linz              |1               |0                 |
|Udaipur           |1               |0                 |
|El Alto           |1               |0                 |
|Oyo               |1               |0                 |
|Juiz de Fora      |1               |0                 |
|Esfahan           |1               |0                 |
|Monywa            |1               |0                 |
|Sultanbeyli       |1               |0                 |
|Dhule (Dhulia)    |1               |0                 |
|Mit Ghamr         |1               |0     

In [108]:
# 7. CATEGORY WITH HIGHEST RENTAL HOURS

rental_hours = (
    rental
    .withColumn(
        "rental_hours",
        (
            F.unix_timestamp("return_date") -
            F.unix_timestamp("rental_date")
        ) / 3600
    )
)

In [109]:
city_category_hours = (
    rental_hours
    .join(customer, on="customer_id", how="inner")
    .join(address, on="address_id", how="inner")
    .join(city, on="city_id", how="inner")
    .join(inventory, on="inventory_id", how="inner")
    .join(film_category, on="film_id", how="inner")
    .join(category, on="category_id", how="inner")
    .groupBy("city", "name")
    .agg(
        F.sum("rental_hours")
        .alias("total_hours")
    )
)

In [110]:
# 7A. CITIES STARTING WITH "a"

a_cities_global = (
    city_category_hours
    .filter(
        F.lower(
            F.col("city")
        ).startswith("a")
    )
    .groupBy("name")
    .agg(
        F.sum("total_hours")
        .alias("global_hours")
    )
    .orderBy(
        F.desc("global_hours")
    )
    .limit(1)
)

print("7A. Top category for cities starting with 'a'")
a_cities_global.show(truncate=False)

7A. Top category for cities starting with 'a'
+------+------------------+
|name  |global_hours      |
+------+------------------+
|Sports|12360.350000000002|
+------+------------------+



In [111]:
# 7B. CITIES CONTAINING "-"

dash_cities_global = (
    city_category_hours
    .filter(
        F.col("city").contains("-")
    )
    .groupBy("name")
    .agg(
        F.sum("total_hours")
        .alias("global_hours")
    )
    .orderBy(
        F.desc("global_hours")
    )
    .limit(1)
)

print("7B. Top category for cities containing '-'")
dash_cities_global.show(truncate=False)

7B. Top category for cities containing '-'
+-------+------------+
|name   |global_hours|
+-------+------------+
|Foreign|6472.15     |
+-------+------------+



In [113]:
# STOP SPARK SESSION
spark.stop()